In [1]:
import random
import math

In [73]:
class Board:

    def __init__(self, to_plant = 4):
        self.tileset_odds = [0.30,0.27,0.23,0.20]
        self.blank = "-"
        self.board_width = 190
        self.board_height = 43
        self.board = []
        self.reset_board()
        self.start_seeds = []
        self.num_to_plant = to_plant
        self.plant_seeds()
        self.germinate()

    def reset_board(self):
        self.board = [[self.blank] * self.board_width for _ in range(self.board_height)]
    
    def get_tileset_choice(self):
        tileset_random = random.random()
        tileset_choice = -1
        
        for i, tso in enumerate(self.tileset_odds):
            tileset_random -= tso
            if tileset_random <= 0 and tileset_choice < 0:
                tileset_choice = i
        return tileset_choice

    def get_random_point(self):
        return (math.floor(random.random() * self.board_width), math.floor(random.random() * self.board_height))

    def plant_seeds(self):
        self.start_seeds = []
        for i in range(self.num_to_plant):
            p = self.get_random_point()
            tsc = self.get_tileset_choice()
            self.board[p[1]][p[0]] = tsc
            self.start_seeds.append((p,tsc))

    def closest_open(self, point):
        min_horiz = point[0]
        max_horiz = point[0]
        min_height = point[1]
        max_height = point[1]
        map_full = False

        open_spots = []
        count = 0

        while len(open_spots) == 0 and not map_full:
            count += 1
            min_horiz = point[0] - count
            max_horiz = point[0] + count
            min_height = point[1] - count
            max_height = point[1] + count
    
            min_horiz = 0 if min_horiz < 0 else min_horiz
            max_horiz = self.board_width - 1 if max_horiz >= self.board_width else max_horiz
            min_height = 0 if min_height < 0 else min_height
            max_height = self.board_height - 1 if max_height >= self.board_height else max_height
            
            for y in range(min_height, max_height+1):
                for x in range(min_horiz, max_horiz+1):
                    if self.board[y][x] == self.blank:
                        open_spots.append((x,y))

            # Searched the entire board and found nothing
            if (min_horiz == 0 and max_horiz == self.board_width - 1 and min_height == 0 and max_height == self.board_height - 1) and len(open_spots) == 0:
                map_full = True

        ret = None
        if not map_full:
            dex = math.floor(random.random() * len(open_spots))
            ret = open_spots[dex]
            
        return ret
        
    def get_filled_adjacent(self, point):
        min_horiz = point[0]
        max_horiz = point[0]
        min_height = point[1]
        max_height = point[1]

        filled_spots = []

        min_horiz = point[0] - 1
        max_horiz = point[0] + 1
        min_height = point[1] - 1
        max_height = point[1] + 1

        min_horiz = 0 if min_horiz < 0 else min_horiz
        max_horiz = self.board_width - 1 if max_horiz >= self.board_width else max_horiz
        min_height = 0 if min_height < 0 else min_height
        max_height = self.board_height - 1 if max_height >= self.board_height else max_height
        
        for y in range(min_height, max_height+1):
            for x in range(min_horiz, max_horiz+1):
                if self.board[y][x] != self.blank:
                    filled_spots.append(self.board[y][x])
        return filled_spots

    def gen_ts_for_p(self, point):
        filledadj = self.get_filled_adjacent(point)
        num_filled = len(filledadj)
        current_set_count = [0 for _ in range(len(self.tileset_odds))]
        ret_ts = None

        for fda in filledadj:
            current_set_count[fda] += 1
        
        max_val = max(current_set_count)
        max_dexs = [i for i, v in enumerate(current_set_count) if v == max_val]

        # Change this to use biomes that have: reach, decay_rate, maximum_influence, and threshold values
        # Reach - The maximum distance that influence can effect tiles
        # Decay Rate - The rate at which the maximum influence drops as you move away from the source. Potential options: Linear, Logarithmic, Exponential, Constant
        # Maximum Influence - The amount of influence at the center point of the biome
        # Threshold - if multiple biomes are competing for the same point then biomes are removed from consideration
        # when they have the lowest value for: threshold - external_influance
        # in the event of a tie between multiple biomes, the point is assigned randomly to a biome based on
        # a probability distribution relative to the amount of tiles of that biome adjacent to the point
        # Ex) Let's say 3 Plains, 2 Jungle, and 1 Mountain biome tile(s) are competing for the same empty space and tie.
        # In that event the space would be assigned randomly with Plains having a 50% chance, Jungle having a 33% chance
        # and Mountain having a 17% chance
        if len(max_dexs) > 1: #if 2 or more adjacent tile sets occupy the same number of surrounding tiles
            #for now, just randomly pick one
            ret_ts = math.floor(random.random() * len(max_dexs))
        elif max_val >= 0: #if only 1 tileset is adjacent and the empty tile has three or more adjacent neighbors, the new tile becomes part of that set
            ret_ts = max_dexs[0]
        else: 
            # If there is only 1 adjacent tile set, but it doesn't occupy enough neighboring spaces to convert the empty tile
            # then assign a tileset at random based on self.tileset_odds
            ret_ts = self.get_tileset_choice()

        return ret_ts
            
    def assign_ts(self, ts, point):
        self.board[point[1]][point[0]] = ts
        
    def germinate(self):
        is_space = True
        while is_space:
            for p in self.start_seeds:
                clopen = self.closest_open(p[0])
                if not clopen is None:
                    ts = self.gen_ts_for_p(clopen)
                    self.assign_ts(ts, clopen)
                else:
                    is_space = False
    
    def print_board(self):
        tileset = ["%","{",".","^"]
        for row in self.board:
            row_str = ""
            for cell in row:
                row_str += tileset[cell]
            print(row_str)

In [75]:
board = Board(20)
board.print_board()

%%%%%%%%%%%%%%%...........................................................{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%^^^^^^^^^^^^^^^^^^{{{{{{{{{{{{{{{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%%...........................................................{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%^^^^^^^^^^^^^^^^^^{{{{{{{{{{{{{{{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%{...........................................................%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%^^^^^^^^^^^^^^^^^^{{{{{{{{{{{{{{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%%..........................................................{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%^^^^^^^^^^^^^^^^^^{{{{{{{{{{{{{{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%%.................................%%............{....{{^%^^^%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%^^^^^^^^^^^^^^^^^^{{{{{{{{{{{{{{%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
%%%%%%%%%%%%%%%{{{{{{{{{{%.................{^